# Caso MPC Cart-Pole — Red Neuronal Float32 y Comparación con MPC

Este notebook corresponde a la **segunda etapa** del caso MPC basado en el repositorio
`hshose/Adaptive-AMPC-Cartpole`.

Objetivos:

1. Cargar `params.yaml` y el checkpoint `model.eqx`.
2. Reconstruir exactamente la arquitectura `AMPCNN` original en JAX/Equinox.
3. Calcular la normalización usada durante el entrenamiento a partir de `X0.csv` y `U.csv`.
4. Evaluar la red sobre muestras del dataset:
   \[
   x_0 \rightarrow u^{NN}
   \]
   y comparar con la primera acción del MPC:
   \[
   x_0 \rightarrow u_0^{MPC}
   \]
5. Ejecutar la red Float32 en lazo cerrado sobre la dinámica Cart-Pole original.
6. Comparar trayectorias:
   - MPC almacenado en `X.csv`, `U.csv`
   - NN Float32 en lazo cerrado.
7. Calcular métricas base que luego se reutilizarán para PTQ, QAT e INT16x8/Mixed Precision.

---

## Corrección importante respecto al notebook anterior

Al revisar el código original del repositorio, el orden correcto del estado es:

\[
x =
[x_\mathrm{cart},\ \theta,\ \dot{x}_\mathrm{cart},\ \dot{\theta}]
\]

es decir:

- índice 0: posición del carro
- índice 1: ángulo del péndulo
- índice 2: velocidad del carro
- índice 3: velocidad angular

Este orden se usa en todo este notebook.

## 0. Dependencias

In [ ]:
# Ejecutar solo si tu entorno todavía no tiene estas librerías.
# En Anaconda/Jupyter puedes usar:
#
# %pip install jax jaxlib equinox pyyaml scipy pandas matplotlib

import sys
print(sys.version)

In [ ]:
from pathlib import Path
import csv
import pickle
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import equinox as eqx

from scipy.integrate import odeint

jax.config.update("jax_platform_name", "cpu")
np.set_printoptions(precision=6, suppress=True)

print("JAX:", jax.__version__)
print("Equinox:", eqx.__version__)
print("Dispositivo JAX:", jax.devices())

## 1. Configuración de rutas

In [ ]:
# ============================================================
# AJUSTA ESTAS RUTAS
# ============================================================

DATASET_DIR = Path(r"C:\RUTA\AL\FinalAMPCPendulumDataset")

X_PATH  = DATASET_DIR / "X.csv"
U_PATH  = DATASET_DIR / "U.csv"
X0_PATH = DATASET_DIR / "X0.csv"

# Carpeta que contiene model.eqx y params.yaml.
MODEL_DIR = Path(r"C:\RUTA\AL\model_8x50\It3")

MODEL_PATH = MODEL_DIR / "model.eqx"
PARAMS_PATH = MODEL_DIR / "params.yaml"

# Si model.eqx y params.yaml están en el mismo directorio del notebook:
# MODEL_DIR = Path(".")
# MODEL_PATH = MODEL_DIR / "model.eqx"
# PARAMS_PATH = MODEL_DIR / "params.yaml"

print("Dataset:", DATASET_DIR)
print("Modelo :", MODEL_PATH)
print("Params :", PARAMS_PATH)

## 2. Verificación de archivos

In [ ]:
for p in [X_PATH, U_PATH, X0_PATH, MODEL_PATH, PARAMS_PATH]:
    print(f"{p.name:12s} ->", "OK" if p.exists() else "NO ENCONTRADO")

if not PARAMS_PATH.exists():
    raise FileNotFoundError("No se encontró params.yaml. Corrige MODEL_DIR.")
if not MODEL_PATH.exists():
    raise FileNotFoundError("No se encontró model.eqx. Corrige MODEL_DIR.")

## 3. Leer parámetros de la red

In [ ]:
with open(PARAMS_PATH, "r", encoding="utf-8") as f:
    params = yaml.safe_load(f)

params

In [ ]:
keys_mostrar = [
    "num_layers",
    "num_neurons",
    "num_sys_states",
    "num_sys_inputs",
    "num_aug_params",
    "activation_function",
]

print("Arquitectura:")
for k in keys_mostrar:
    print(f"{k:22s}: {params.get(k)}")

print("\nInterpretación del repositorio:")
print("- Entrada: 4 estados")
print("- Salida: 1 acción de control")
print("- Activación oculta: tanh")
print("- Capa de salida: lineal")

## 4. Reconstrucción exacta de la arquitectura Equinox

El repositorio implementa cada capa como:

\[
y = W x + b
\]

seguida por `tanh` en las capas ocultas. La última capa es lineal.

Para poder deserializar `model.eqx`, Equinox necesita primero construir un árbol de módulos con
**exactamente la misma estructura** que tenía el modelo al guardarse.

In [ ]:
import math

def init_weight(dim_in, dim_out, key):
    stdv = 1.0 / math.sqrt(dim_out)
    return jax.random.uniform(key, (dim_out, dim_in)) * 2 * stdv - stdv

def init_bias(dim_out, key):
    stdv = 1.0 / math.sqrt(dim_out)
    return jax.random.uniform(key, (dim_out,)) * 2 * stdv - stdv


class FullyConnectedLayer(eqx.Module):
    weight: jax.Array
    bias: jax.Array
    activation: callable
    activation_function_name: str
    input_dim: int
    output_dim: int

    def __init__(self, input_dim, output_dim, rng_key, activation_function="linear"):
        key_weight, key_bias = jax.random.split(rng_key)

        self.weight = init_weight(input_dim, output_dim, key_weight)
        self.bias = init_bias(output_dim, key_bias)

        self.input_dim = input_dim
        self.output_dim = output_dim

        if activation_function == "linear":
            self.activation = None
        elif activation_function == "relu":
            self.activation = jax.nn.relu
        elif activation_function == "tanh":
            self.activation = jax.nn.tanh
        else:
            raise ValueError(f"Activación no soportada: {activation_function}")

        self.activation_function_name = activation_function

    def __call__(self, x):
        y = self.weight @ x + self.bias
        if self.activation is None:
            return y
        return self.activation(y)


class MLP(eqx.Module):
    layers: list

    def __init__(self, layer_structure, rng_key, activation_function="linear"):
        if len(layer_structure) < 2:
            self.layers = []
            return

        keys = jax.random.split(rng_key, num=len(layer_structure) - 1)

        self.layers = [
            FullyConnectedLayer(
                input_dim=layer_structure[0],
                output_dim=layer_structure[1],
                rng_key=keys[0],
                activation_function=activation_function,
            )
        ]

        for i in range(1, len(layer_structure) - 1):
            self.layers.append(
                FullyConnectedLayer(
                    input_dim=layer_structure[i],
                    output_dim=layer_structure[i + 1],
                    rng_key=keys[i],
                    activation_function=(
                        activation_function
                        if i < len(layer_structure) - 2
                        else "linear"
                    ),
                )
            )

    def __call__(self, x):
        def call_mlp(x_):
            for layer in self.layers:
                x_ = layer(x_)
            return x_

        if len(x.shape) == 2:
            return jax.vmap(call_mlp)(x)

        return call_mlp(x)


class AMPCNN(eqx.Module):
    output_dim: int
    num_sys_states: int
    num_sys_inputs: int
    num_aug_params: int
    neural_network: object

    def __init__(
        self,
        num_layers,
        num_neurons,
        num_sys_states,
        num_sys_inputs,
        num_aug_params,
        rng_key,
        activation_function="tanh",
    ):
        self.num_sys_states = num_sys_states
        self.num_sys_inputs = num_sys_inputs
        self.num_aug_params = num_aug_params
        self.output_dim = num_sys_inputs

        layer_structure = [num_sys_states]

        for _ in range(num_layers - 1):
            layer_structure.append(num_neurons)

        layer_structure.append(self.output_dim)

        self.neural_network = MLP(
            layer_structure,
            rng_key,
            activation_function=activation_function,
        )

    @eqx.filter_jit
    def __call__(self, x):
        return self.neural_network(x)

## 5. Cargar `model.eqx`

In [ ]:
init_key = jax.random.PRNGKey(1)

model_template = AMPCNN(
    num_layers=params["num_layers"],
    num_neurons=params["num_neurons"],
    num_sys_states=params["num_sys_states"],
    num_sys_inputs=params["num_sys_inputs"],
    num_aug_params=params["num_aug_params"],
    rng_key=init_key,
    activation_function=params["activation_function"],
)

model = eqx.tree_deserialise_leaves(str(MODEL_PATH), model_template)

print("Modelo cargado correctamente.")
print(model)

## 6. Inspección de pesos y número de parámetros

In [ ]:
num_params = 0

for i, layer in enumerate(model.neural_network.layers):
    W = np.asarray(layer.weight)
    b = np.asarray(layer.bias)

    n = W.size + b.size
    num_params += n

    print(
        f"Capa {i:02d}: "
        f"W{W.shape}, b{b.shape}, "
        f"activación={layer.activation_function_name}, "
        f"parámetros={n}"
    )

print("\nTotal de parámetros:", num_params)
print("Memoria teórica Float32 de parámetros:", num_params * 4, "bytes")

## 7. Normalización utilizada por los autores

El entrenamiento original calcula:

\[
x_\mathrm{norm} =
\frac{x-\mu_x}{s_x}
\]

con:

\[
\mu_x = \mathrm{mean}(X_0), \qquad
s_x = \max(|X_0|)
\]

Para la acción de control:

\[
u_\mathrm{norm} =
\frac{u-\mu_u}{s_u}
\]

El código original usa **solo la primera acción de cada solución MPC** para entrenar el controlador explícito:

```python
u_data = U[:, 0:num_sys_inputs]
```

Por ello calcularemos la normalización directamente desde `X0.csv` y la primera columna de `U.csv`.

Se hace mediante lectura por bloques para no cargar varios GB en RAM.

In [ ]:
def estadisticas_media_maxabs_csv(path, columnas=None, chunksize=50000):
    suma = None
    maxabs = None
    n = 0

    for chunk in pd.read_csv(
        path,
        header=None,
        usecols=columnas,
        chunksize=chunksize,
        dtype=np.float64,
    ):
        a = chunk.to_numpy()

        if suma is None:
            suma = np.zeros(a.shape[1], dtype=np.float64)
            maxabs = np.zeros(a.shape[1], dtype=np.float64)

        suma += a.sum(axis=0)
        maxabs = np.maximum(maxabs, np.max(np.abs(a), axis=0))
        n += len(a)

    return suma / n, maxabs, n

In [ ]:
print("Calculando normalización de X0...")
mu_x, scale_x, n_x = estadisticas_media_maxabs_csv(
    X0_PATH,
    columnas=[0, 1, 2, 3],
)

print("Calculando normalización de U0...")
mu_u, scale_u, n_u = estadisticas_media_maxabs_csv(
    U_PATH,
    columnas=[0],
)

normalization = {
    "x": (mu_x, scale_x),
    "u": (mu_u, scale_u),
}

print("\nNúmero de muestras X0:", n_x)
print("Número de muestras U0:", n_u)

print("\nmu_x    =", mu_x)
print("scale_x =", scale_x)
print("mu_u    =", mu_u)
print("scale_u =", scale_u)

### Guardar la normalización

Esto evita tener que volver a recorrer los CSV grandes cada vez que abras el notebook.

In [ ]:
NORMALIZATION_CACHE = MODEL_DIR / "normalization_reconstructed.pkl"

with open(NORMALIZATION_CACHE, "wb") as f:
    pickle.dump(normalization, f)

print("Guardado en:", NORMALIZATION_CACHE)

Si ya ejecutaste el cálculo previamente, en ejecuciones futuras puedes omitir las dos celdas anteriores y cargar:

In [ ]:
# with open(NORMALIZATION_CACHE, "rb") as f:
#     normalization = pickle.load(f)
#
# normalization

## 8. Función de inferencia física de la red

In [ ]:
def normalize(data, normalization_params):
    return (data - normalization_params[0]) / normalization_params[1]

def denormalize(data, normalization_params):
    return data * normalization_params[1] + normalization_params[0]


def controlador_nn_float32(x, model=model, normalization=normalization):
    x = np.asarray(x, dtype=np.float32)

    x_norm = normalize(x, normalization["x"])
    x_norm = jnp.asarray(x_norm, dtype=jnp.float32)

    u_norm = model(x_norm)

    u = denormalize(
        np.asarray(u_norm, dtype=np.float32),
        normalization["u"]
    )

    # Saturación usada en el código original.
    u = np.clip(u, -9.0, 9.0)

    return np.asarray(u, dtype=np.float32)

In [ ]:
# Prueba en el equilibrio.
x_eq = np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32)

u_eq = controlador_nn_float32(x_eq)

print("x =", x_eq)
print("u_NN =", u_eq)

## 9. Cargar muestras del dataset para comparación estática

Aquí compararemos:

\[
u_0^{MPC}
\quad\text{vs}\quad
u^{NN}(x_0)
\]

Esta es una comparación de **aproximación de política**, no todavía una prueba de lazo cerrado.

In [ ]:
def cargar_primeras_acciones(nrows=10000):
    X0_sample = pd.read_csv(
        X0_PATH,
        header=None,
        nrows=nrows,
        dtype=np.float32,
    ).to_numpy()

    U0_sample = pd.read_csv(
        U_PATH,
        header=None,
        usecols=[0],
        nrows=nrows,
        dtype=np.float32,
    ).to_numpy().reshape(-1)

    return X0_sample, U0_sample


N_STATIC = 10000

X0_static, U0_mpc_static = cargar_primeras_acciones(N_STATIC)

print("X0:", X0_static.shape)
print("U0:", U0_mpc_static.shape)

In [ ]:
def inferir_batch_nn(X_batch):
    X_batch = np.asarray(X_batch, dtype=np.float32)

    X_norm = normalize(X_batch, normalization["x"]).astype(np.float32)
    U_norm = model(jnp.asarray(X_norm))

    U = denormalize(
        np.asarray(U_norm, dtype=np.float32),
        normalization["u"]
    )

    return np.clip(U.reshape(-1), -9.0, 9.0)


U0_nn_static = inferir_batch_nn(X0_static)

print(U0_nn_static[:10])

## 10. Métricas estáticas MPC vs NN Float32

In [ ]:
def rmse(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    return float(np.sqrt(np.mean((a - b) ** 2)))

def mae(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    return float(np.mean(np.abs(a - b)))

def max_abs_error(a, b):
    return float(np.max(np.abs(np.asarray(a) - np.asarray(b))))

def r2_score_np(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    return float(1.0 - ss_res / ss_tot)


metricas_estaticas = {
    "RMSE_u": rmse(U0_mpc_static, U0_nn_static),
    "MAE_u": mae(U0_mpc_static, U0_nn_static),
    "MaxAbsError_u": max_abs_error(U0_mpc_static, U0_nn_static),
    "R2_u": r2_score_np(U0_mpc_static, U0_nn_static),
}

pd.DataFrame([metricas_estaticas], index=["NN Float32"])

## 11. Gráficos de aproximación estática

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(U0_mpc_static, U0_nn_static, s=8, alpha=0.35)

lim_min = min(U0_mpc_static.min(), U0_nn_static.min())
lim_max = max(U0_mpc_static.max(), U0_nn_static.max())

plt.plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--")
plt.xlabel("u MPC")
plt.ylabel("u NN Float32")
plt.title("Acción de control — MPC vs NN Float32")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
errores_u = U0_nn_static - U0_mpc_static

plt.figure(figsize=(9, 4))
plt.hist(errores_u, bins=100)
plt.xlabel(r"$u_{NN} - u_{MPC}$")
plt.ylabel("Frecuencia")
plt.title("Distribución del error de aproximación Float32")
plt.grid(True, alpha=0.3)
plt.show()

print("Media del error:", errores_u.mean())
print("STD del error  :", errores_u.std())

# 12. Dinámica Cart-Pole original

La dinámica usada por los autores tiene estado:

\[
x =
[x_\mathrm{cart},\theta,\dot{x}_\mathrm{cart},\dot{\theta}]
\]

y el controlador entrega una tensión/comando \(V_m\).

Se replica el modelo identificado incluido en el repositorio.

In [ ]:
# Parámetros identificados utilizados por los autores.

M_cart_opt = 0.506
m_rod_opt = 0.23
L_rod_opt = 0.6393037858519218
J_mot_eq_opt = 0.2153426947227305

AB_opt = -3.9610232930789304
AC_opt = 1.3002429170001302
B_eq_opt = 3.961023945944119
B_p_opt = 0.0002073799633807001


def f_cartpole(
    x,
    u,
    m_add=0.02,
    M_cart=M_cart_opt,
    m_rod=m_rod_opt,
    L_rod=L_rod_opt,
    J_mot_eq=J_mot_eq_opt,
    AB=AB_opt,
    AC=AC_opt,
    B_eq=B_eq_opt,
    Bp=B_p_opt,
):
    g = 9.81
    L = L_rod

    m = m_add + m_rod
    l = (L / 2 * m_rod + L * m_add) / m

    J_rod = (m_rod * L**2) / 12
    J = J_rod + m_add * L**2

    M = M_cart + J_mot_eq

    h1 = M + m
    h2 = m * l
    h4 = m * l**2 + J
    h7 = m * l * g

    theta = x[1]
    v1 = x[2]
    dtheta = x[3]

    Vm = float(np.asarray(u).reshape(-1)[0])

    F = AB * v1 + AC * Vm

    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)

    denominator = h2**2 * cos_theta**2 - h1 * h4

    dx = np.array([
        v1,
        dtheta,
        (
            h2 * h4 * dtheta**2 * sin_theta
            - h2 * h7 * cos_theta * sin_theta
            + h4 * (F - B_eq * v1)
            - h2 * cos_theta * Bp * dtheta
        ) / (-denominator),
        (
            h2**2 * dtheta**2 * cos_theta * sin_theta
            - h1 * h7 * sin_theta
            + h2 * cos_theta * (F - B_eq * v1)
            + h1 * Bp * dtheta
        ) / denominator,
    ], dtype=np.float64)

    return dx

## 13. Simulador con entrada constante por periodo de muestreo

In [ ]:
DT = 160e-3   # 160 ms, igual al código de evaluación original.

def simular_un_paso(x, u, dt=DT):
    def ode_rhs(y, t):
        return f_cartpole(y, u)

    traj = odeint(
        ode_rhs,
        np.asarray(x, dtype=np.float64),
        np.linspace(0.0, dt, 2),
    )

    return traj[-1]

## 14. Cargar una trayectoria MPC completa

In [ ]:
def cargar_fila_csv(path, idx, dtype=np.float64):
    with Path(path).open("r", newline="", encoding="utf-8") as f:
        reader = csv.reader(f)

        for i, row in enumerate(reader):
            if i == idx:
                return np.asarray(row, dtype=dtype)

    raise IndexError(f"No existe la fila {idx} en {path}")


def cargar_trayectoria_mpc(idx):
    x0 = cargar_fila_csv(X0_PATH, idx)
    x_flat = cargar_fila_csv(X_PATH, idx)
    u = cargar_fila_csv(U_PATH, idx)

    X_mpc = x_flat.reshape(-1, 4)

    return x0, X_mpc, u


TRAYECTORIA_ID = 0

x0_ref, X_mpc, U_mpc = cargar_trayectoria_mpc(TRAYECTORIA_ID)

print("x0       :", x0_ref)
print("X_mpc    :", X_mpc.shape)
print("U_mpc    :", U_mpc.shape)
print("N MPC     :", len(U_mpc))

## 15. Closed-loop con NN Float32

La red se evalúa en cada paso usando el estado realimentado:

\[
u_k^{NN}=\pi_{NN}(x_k)
\]

\[
x_{k+1}=f(x_k,u_k^{NN})
\]

Para poder comparar directamente con la trayectoria almacenada del MPC, simulamos inicialmente
el mismo número de pasos que el horizonte de `U.csv`.

In [ ]:
def simular_closed_loop_nn(x0, n_steps, dt=DT):
    X_nn = np.zeros((n_steps + 1, 4), dtype=np.float64)
    U_nn = np.zeros(n_steps, dtype=np.float64)

    X_nn[0] = np.asarray(x0, dtype=np.float64)

    for k in range(n_steps):
        u = controlador_nn_float32(X_nn[k])

        U_nn[k] = float(np.asarray(u).reshape(-1)[0])
        X_nn[k + 1] = simular_un_paso(X_nn[k], [U_nn[k]], dt=dt)

    return X_nn, U_nn


X_nn, U_nn = simular_closed_loop_nn(
    x0_ref,
    n_steps=len(U_mpc),
    dt=DT,
)

print("X_nn:", X_nn.shape)
print("U_nn:", U_nn.shape)
print("\nEstado inicial NN:", X_nn[0])
print("Estado final NN  :", X_nn[-1])

## 16. Comparación MPC vs NN Float32

In [ ]:
STATE_LABELS = [
    "Posición del carro x [m]",
    "Ángulo θ [rad]",
    "Velocidad del carro ẋ [m/s]",
    "Velocidad angular θ̇ [rad/s]",
]

t_x = np.arange(X_mpc.shape[0]) * DT
t_u = np.arange(U_mpc.shape[0]) * DT

for i, label in enumerate(STATE_LABELS):
    plt.figure(figsize=(10, 4))

    plt.plot(t_x, X_mpc[:, i], label="MPC")
    plt.plot(t_x, X_nn[:, i], linestyle="--", label="NN Float32")

    plt.axhline(0, linewidth=1)
    plt.xlabel("Tiempo [s]")
    plt.ylabel(label)
    plt.title(f"MPC vs NN Float32 — {label}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

plt.step(t_u, U_mpc, where="post", label="MPC")
plt.step(t_u, U_nn, where="post", linestyle="--", label="NN Float32")

plt.axhline(0, linewidth=1)
plt.axhline(9, linestyle=":", linewidth=1)
plt.axhline(-9, linestyle=":", linewidth=1)

plt.xlabel("Tiempo [s]")
plt.ylabel("u")
plt.title("MPC vs NN Float32 — Acción de control")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 17. Error dinámico de estados

In [ ]:
errores_estado = X_nn - X_mpc

for i, label in enumerate(STATE_LABELS):
    plt.figure(figsize=(9, 4))
    plt.plot(t_x, errores_estado[:, i])
    plt.axhline(0, linewidth=1)
    plt.xlabel("Tiempo [s]")
    plt.ylabel("NN - MPC")
    plt.title(f"Error dinámico — {label}")
    plt.grid(True, alpha=0.3)
    plt.show()

## 18. Métricas dinámicas

In [ ]:
def variacion_total(u):
    return float(np.sum(np.abs(np.diff(u))))

def esfuerzo_control(u):
    return float(np.sum(np.asarray(u) ** 2))

metricas_dinamicas = {
    "RMSE_x_cart": rmse(X_mpc[:, 0], X_nn[:, 0]),
    "RMSE_theta": rmse(X_mpc[:, 1], X_nn[:, 1]),
    "RMSE_xdot": rmse(X_mpc[:, 2], X_nn[:, 2]),
    "RMSE_thetadot": rmse(X_mpc[:, 3], X_nn[:, 3]),
    "RMSE_u": rmse(U_mpc, U_nn),
    "MAE_u": mae(U_mpc, U_nn),
    "MaxAbsError_u": max_abs_error(U_mpc, U_nn),
    "TV_u_MPC": variacion_total(U_mpc),
    "TV_u_NN": variacion_total(U_nn),
    "Esfuerzo_MPC": esfuerzo_control(U_mpc),
    "Esfuerzo_NN": esfuerzo_control(U_nn),
    "Norma_estado_final_MPC": float(np.linalg.norm(X_mpc[-1])),
    "Norma_estado_final_NN": float(np.linalg.norm(X_nn[-1])),
}

pd.DataFrame([metricas_dinamicas], index=["NN Float32"])

## 19. Criterio de estabilización

El repositorio evalúa el estado final mediante ventanas de las últimas muestras.

Aquí definimos un criterio más interpretable para la tesis:

\[
|x| < \epsilon_x,\quad
|\theta| < \epsilon_\theta,\quad
|\dot{x}| < \epsilon_{\dot{x}},\quad
|\dot{\theta}| < \epsilon_{\dot{\theta}}
\]

durante todas las muestras restantes.

Los umbrales siguientes son configurables.

In [ ]:
THRESHOLDS = np.array([
    0.10,   # posición [m]
    0.17,   # ángulo [rad]
    0.10,   # velocidad [m/s]
    0.17,   # velocidad angular [rad/s]
])

def tiempo_estabilizacion(X, thresholds=THRESHOLDS, dt=DT):
    dentro = np.all(np.abs(X) <= thresholds, axis=1)

    for k in range(len(dentro)):
        if np.all(dentro[k:]):
            return k * dt

    return np.nan


print(
    "Tiempo de estabilización MPC:",
    tiempo_estabilizacion(X_mpc),
    "s"
)

print(
    "Tiempo de estabilización NN Float32:",
    tiempo_estabilizacion(X_nn),
    "s"
)

## 20. Closed-loop más largo con la NN

In [ ]:
# El dataset MPC contiene N=25 pasos, pero el código de testing de los autores
# ejecuta pruebas más largas. Esto permite observar si la red realmente mantiene
# el equilibrio después del horizonte almacenado.

N_LONG = 60

X_nn_long, U_nn_long = simular_closed_loop_nn(
    x0_ref,
    n_steps=N_LONG,
    dt=DT,
)

t_long_x = np.arange(N_LONG + 1) * DT
t_long_u = np.arange(N_LONG) * DT

for i, label in enumerate(STATE_LABELS):
    plt.figure(figsize=(10, 4))
    plt.plot(t_long_x, X_nn_long[:, i])
    plt.axhline(0, linewidth=1)
    plt.xlabel("Tiempo [s]")
    plt.ylabel(label)
    plt.title(f"NN Float32 closed-loop extendido — {label}")
    plt.grid(True, alpha=0.3)
    plt.show()

plt.figure(figsize=(10, 4))
plt.step(t_long_u, U_nn_long, where="post")
plt.axhline(0, linewidth=1)
plt.axhline(9, linestyle=":", linewidth=1)
plt.axhline(-9, linestyle=":", linewidth=1)
plt.xlabel("Tiempo [s]")
plt.ylabel("u")
plt.title("NN Float32 closed-loop extendido — Acción de control")
plt.grid(True, alpha=0.3)
plt.show()

## 21. Evaluación de múltiples condiciones iniciales

In [ ]:
IDS_EVALUACION = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

resumen = []

for idx in IDS_EVALUACION:
    x0_i, X_mpc_i, U_mpc_i = cargar_trayectoria_mpc(idx)

    X_nn_i, U_nn_i = simular_closed_loop_nn(
        x0_i,
        n_steps=len(U_mpc_i),
        dt=DT,
    )

    resumen.append({
        "trayectoria": idx,
        "RMSE_x": rmse(X_mpc_i[:, 0], X_nn_i[:, 0]),
        "RMSE_theta": rmse(X_mpc_i[:, 1], X_nn_i[:, 1]),
        "RMSE_xdot": rmse(X_mpc_i[:, 2], X_nn_i[:, 2]),
        "RMSE_thetadot": rmse(X_mpc_i[:, 3], X_nn_i[:, 3]),
        "RMSE_u": rmse(U_mpc_i, U_nn_i),
        "MAE_u": mae(U_mpc_i, U_nn_i),
        "TV_u_NN": variacion_total(U_nn_i),
        "norma_final_NN": np.linalg.norm(X_nn_i[-1]),
        "t_est_NN_s": tiempo_estabilizacion(X_nn_i),
    })

df_multi = pd.DataFrame(resumen).set_index("trayectoria")
display(df_multi)

In [ ]:
print("Promedios:")
display(df_multi.mean(numeric_only=True).to_frame("media").T)

print("Peor caso por métrica:")
display(df_multi.max(numeric_only=True).to_frame("máximo").T)

# 22. Resultado base para las futuras cuantizaciones

Una vez verificado este notebook, la red Float32 queda establecida como **baseline neuronal**.

El flujo posterior será:

```text
MPC experto
    │
    ├── NN Float32  ← baseline de este notebook
    │
    ├── PTQ INT8
    │
    ├── QAT INT8
    │
    ├── INT16x8
    │
    └── Mixed Precision
```

Para cada variante se repetirán **exactamente**:

1. evaluación estática:
   - RMSE de \(u\)
   - MAE
   - error máximo
2. evaluación closed-loop:
   - RMSE de los cuatro estados
   - estabilidad
   - tiempo de estabilización
   - esfuerzo de control
   - variación total de \(u\)
   - saturaciones
3. hardware STM32N6570-DK:
   - Flash
   - SRAM
   - latencia
   - backend de ejecución

# 23. Validación de la dinámica — Replay MPC vs `X.csv`

Antes de atribuir las diferencias entre MPC y NN Float32 al error de aproximación de la red,
se debe comprobar que la dinámica Cart-Pole reconstruida en este notebook reproduce correctamente
las trayectorias almacenadas en el dataset.

La prueba consiste en tomar:

- la condición inicial \(x_0\) desde `X0.csv`;
- la secuencia óptima \(U_{MPC}=[u_0,\ldots,u_{N-1}]\) desde `U.csv`;

y aplicar exactamente esos controles a la dinámica:

\[
x_{k+1}^{replay}=f(x_k^{replay},u_k^{MPC})
\]

Luego se compara:

\[
X_{replay}\quad \text{vs}\quad X_{dataset}
\]

**Interpretación:**

- Si coinciden estrechamente, la dinámica del notebook es consistente con el dataset.
- Si difieren significativamente, todavía no se puede atribuir todo el error observado a la red neuronal.

In [ ]:
def replay_mpc_desde_dataset(x0, u_mpc, dt=DT):
    u_mpc = np.asarray(u_mpc, dtype=np.float64).reshape(-1)

    X_replay = np.zeros((len(u_mpc) + 1, 4), dtype=np.float64)
    X_replay[0] = np.asarray(x0, dtype=np.float64)

    for k, u_k in enumerate(u_mpc):
        X_replay[k + 1] = simular_un_paso(
            X_replay[k],
            [u_k],
            dt=dt,
        )

    return X_replay

## 23.1 Replay de una trayectoria

In [ ]:
X_replay = replay_mpc_desde_dataset(
    x0_ref,
    U_mpc,
    dt=DT,
)

print("X_dataset:", X_mpc.shape)
print("X_replay :", X_replay.shape)

print("\nEstado inicial dataset:", X_mpc[0])
print("Estado inicial replay :", X_replay[0])

print("\nEstado final dataset:")
print(X_mpc[-1])

print("\nEstado final replay:")
print(X_replay[-1])

print("\nDiferencia final replay - dataset:")
print(X_replay[-1] - X_mpc[-1])

## 23.2 Gráficos `X.csv` vs Replay MPC

In [ ]:
for i, label in enumerate(STATE_LABELS):
    plt.figure(figsize=(10, 4))

    plt.plot(t_x, X_mpc[:, i], label="X.csv — trayectoria MPC")
    plt.plot(t_x, X_replay[:, i], linestyle="--", label="Replay usando U.csv")

    plt.axhline(0, linewidth=1)
    plt.xlabel("Tiempo [s]")
    plt.ylabel(label)
    plt.title(f"Validación dinámica — {label}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

## 23.3 Error de reconstrucción por estado

In [ ]:
error_replay = X_replay - X_mpc

for i, label in enumerate(STATE_LABELS):
    plt.figure(figsize=(9, 4))
    plt.plot(t_x, error_replay[:, i])
    plt.axhline(0, linewidth=1)
    plt.xlabel("Tiempo [s]")
    plt.ylabel("Replay - X.csv")
    plt.title(f"Error de reconstrucción dinámica — {label}")
    plt.grid(True, alpha=0.3)
    plt.show()

## 23.4 Métricas del Replay MPC

In [ ]:
metricas_replay = {
    "RMSE_x_cart": rmse(X_mpc[:, 0], X_replay[:, 0]),
    "RMSE_theta": rmse(X_mpc[:, 1], X_replay[:, 1]),
    "RMSE_xdot": rmse(X_mpc[:, 2], X_replay[:, 2]),
    "RMSE_thetadot": rmse(X_mpc[:, 3], X_replay[:, 3]),
    "MAE_global_estados": mae(X_mpc, X_replay),
    "MaxAbsError_global": max_abs_error(X_mpc, X_replay),
    "Norma_error_final": float(np.linalg.norm(X_replay[-1] - X_mpc[-1])),
}

df_replay = pd.DataFrame([metricas_replay], index=["Replay MPC"])
display(df_replay)

## 23.5 Error relativo respecto al rango de cada estado

Se calcula:

\[
NRMSE_i=
\frac{RMSE_i}{\max(X_i)-\min(X_i)}
\]

para evitar interpretar igual errores absolutos en variables con escalas muy distintas.

In [ ]:
nrmse_replay = {}

for i, label in enumerate(STATE_LABELS):
    rango = np.max(X_mpc[:, i]) - np.min(X_mpc[:, i])
    valor_rmse = rmse(X_mpc[:, i], X_replay[:, i])

    nrmse_replay[label] = valor_rmse / rango if rango > 0 else np.nan

df_nrmse_replay = pd.DataFrame(
    [nrmse_replay],
    index=["NRMSE Replay"]
)

display(df_nrmse_replay)

## 23.6 Comparación directa: MPC dataset vs Replay vs NN Float32

In [ ]:
for i, label in enumerate(STATE_LABELS):
    plt.figure(figsize=(10, 4))

    plt.plot(t_x, X_mpc[:, i], label="MPC dataset")
    plt.plot(t_x, X_replay[:, i], linestyle=":", linewidth=2, label="Replay MPC")
    plt.plot(t_x, X_nn[:, i], linestyle="--", label="NN Float32")

    plt.axhline(0, linewidth=1)
    plt.xlabel("Tiempo [s]")
    plt.ylabel(label)
    plt.title(f"MPC dataset vs Replay vs NN Float32 — {label}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

## 23.7 Diagnóstico automático

Criterio práctico de diagnóstico:

- **NRMSE máximo < 1 %**: reconstrucción muy consistente.
- **1–5 %**: diferencia pequeña/moderada; revisar integración y parámetros.
- **> 5 %**: la dinámica reconstruida probablemente no coincide suficientemente con la usada para generar el dataset.

Estos umbrales son criterios prácticos para esta tesis y **no provienen del paper original**.

In [ ]:
nrmse_vals = np.array(list(nrmse_replay.values()), dtype=np.float64)
max_nrmse = np.nanmax(nrmse_vals)

print(f"NRMSE máximo Replay: {100*max_nrmse:.4f} %")

if max_nrmse < 0.01:
    print(
        "✓ La dinámica reconstruida es muy consistente con X.csv.\n"
        "  Las diferencias NN vs MPC pueden atribuirse principalmente "
        "a la aproximación neuronal."
    )
elif max_nrmse < 0.05:
    print(
        "⚠ El replay presenta diferencias pequeñas/moderadas.\n"
        "  Conviene revisar parámetros, integración y periodo de muestreo."
    )
else:
    print(
        "✗ El replay difiere significativamente de X.csv.\n"
        "  No conviene cuantizar todavía: primero hay que alinear "
        "la dinámica del notebook con la utilizada para generar el dataset."
    )

## 23.8 Validación del Replay sobre múltiples trayectorias

In [ ]:
IDS_REPLAY = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

replay_multi = []

for idx in IDS_REPLAY:
    x0_i, X_mpc_i, U_mpc_i = cargar_trayectoria_mpc(idx)

    X_replay_i = replay_mpc_desde_dataset(
        x0_i,
        U_mpc_i,
        dt=DT,
    )

    replay_multi.append({
        "trayectoria": idx,
        "RMSE_x": rmse(X_mpc_i[:, 0], X_replay_i[:, 0]),
        "RMSE_theta": rmse(X_mpc_i[:, 1], X_replay_i[:, 1]),
        "RMSE_xdot": rmse(X_mpc_i[:, 2], X_replay_i[:, 2]),
        "RMSE_thetadot": rmse(X_mpc_i[:, 3], X_replay_i[:, 3]),
        "MAE_global": mae(X_mpc_i, X_replay_i),
        "MaxAbsError": max_abs_error(X_mpc_i, X_replay_i),
        "Norma_error_final": np.linalg.norm(X_replay_i[-1] - X_mpc_i[-1]),
    })

df_replay_multi = pd.DataFrame(replay_multi).set_index("trayectoria")
display(df_replay_multi)

In [ ]:
print("Promedio del Replay sobre las trayectorias seleccionadas:")
display(df_replay_multi.mean(numeric_only=True).to_frame("media").T)

print("\nPeor caso:")
display(df_replay_multi.max(numeric_only=True).to_frame("máximo").T)

## 23.9 Decisión antes de cuantizar

Secuencia recomendada:

```text
Replay MPC reproduce X.csv
        │
        ├── SÍ → validar NN Float32 en muchas condiciones iniciales
        │           ↓
        │        congelar baseline
        │           ↓
        │        cuantización
        │
        └── NO → revisar dinámica / Ts / parámetros / integración
                    ↓
                 repetir Replay
```

La cuantización debe comenzar después de confirmar que el error NN vs MPC no está siendo contaminado significativamente por una reconstrucción incorrecta de la planta.